# Meteologica Load Forecast — DA Cutoff Revision Analysis (v2)

Inspects how Meteologica PJM demand forecasts for the next 7 days evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 12h / 24h / 48h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates and regions.

## 1. Setup & Data Pull

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
# Load the main cutoff analysis query
sql_path = Path.cwd() / "meteologica_load_forecast_da_cutoff.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
print(f"Regions: {sorted(df['region'].unique())}")
df.head(10)

744 rows
Date range: 2026-03-04 to 2026-03-11
Regions: ['MIDATL', 'RTO', 'SOUTH', 'WEST']


,forecast_date,hour_ending,region,cutoff_load_mw,cutoff_execution_dt,exec_dt_12h,load_mw_12h,exec_dt_24h,load_mw_24h,exec_dt_48h,load_mw_48h,delta_12h,delta_24h,delta_48h
0,2026-03-11,1,MIDATL,21933.0,2026-03-04 08:22:13,2026-03-03 20:20:16,21918.0,2026-03-03 08:20:56,22077.0,2026-02-27 12:19:37,22702.0,15.0,-144.0,-769.0
1,2026-03-11,2,MIDATL,23315.0,2026-03-04 08:22:13,2026-03-03 20:20:16,23309.0,2026-03-03 08:20:56,23470.0,2026-02-27 12:19:37,24166.0,6.0,-155.0,-851.0
2,2026-03-11,3,MIDATL,25727.0,2026-03-04 08:22:13,2026-03-03 20:20:16,25720.0,2026-03-03 08:20:56,25872.0,2026-02-27 12:19:37,26622.0,7.0,-145.0,-895.0
3,2026-03-11,4,MIDATL,27243.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27243.0,2026-03-03 08:20:56,27357.0,2026-02-27 12:19:37,28154.0,0.0,-114.0,-911.0
4,2026-03-11,5,MIDATL,27637.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27680.0,2026-03-03 08:20:56,27606.0,2026-02-27 12:19:37,28584.0,-43.0,31.0,-947.0
5,2026-03-11,6,MIDATL,27585.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27748.0,2026-03-03 08:20:56,27429.0,2026-02-27 12:19:37,28557.0,-163.0,156.0,-972.0
6,2026-03-11,7,MIDATL,27316.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27556.0,2026-03-03 08:20:56,27017.0,2026-02-27 12:19:37,28312.0,-240.0,299.0,-996.0
7,2026-03-11,8,MIDATL,27154.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27390.0,2026-03-03 08:20:56,26763.0,2026-02-27 12:19:37,28100.0,-236.0,391.0,-946.0
8,2026-03-11,9,MIDATL,27140.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27364.0,2026-03-03 08:20:56,26690.0,2026-02-27 12:19:37,27967.0,-224.0,450.0,-827.0
9,2026-03-11,10,MIDATL,27261.0,2026-03-04 08:22:13,2026-03-03 20:20:16,27413.0,2026-03-03 08:20:56,26791.0,2026-02-27 12:19:37,27986.0,-152.0,470.0,-725.0


In [3]:
df.dtypes

forecast_date                  object
hour_ending                     int64
region                            str
cutoff_load_mw                float64
cutoff_execution_dt    datetime64[us]
exec_dt_12h            datetime64[us]
load_mw_12h                   float64
exec_dt_24h            datetime64[us]
load_mw_24h                   float64
exec_dt_48h            datetime64[us]
load_mw_48h                   float64
delta_12h                     float64
delta_24h                     float64
delta_48h                     float64
dtype: object

## 1b. Forecast Coverage Summary

One row per forecast date showing the cutoff vintage used, each lookback vintage,
and peak cutoff load — a quick reference for which days have full 48h→12h history.

In [4]:
# Build a per-forecast-date summary table
_fmt_dt = lambda s: pd.to_datetime(s).strftime("%Y-%m-%d %H:%M") if pd.notna(s) else "—"

forecast_summary = (
    df.groupby("forecast_date")
    .agg(
        regions=("region", lambda x: ", ".join(sorted(x.unique()))),
        n_regions=("region", "nunique"),
        cutoff_exec=("cutoff_execution_dt", "first"),
        exec_12h=("exec_dt_12h", "first"),
        exec_24h=("exec_dt_24h", "first"),
        exec_48h=("exec_dt_48h", "first"),
        peak_cutoff_mw=("cutoff_load_mw", "max"),
        min_cutoff_mw=("cutoff_load_mw", "min"),
        hours_covered=("hour_ending", "nunique"),
    )
    .reset_index()
    .sort_values("forecast_date")
)

# Days from today
forecast_summary["days_ahead"] = (
    pd.to_datetime(forecast_summary["forecast_date"]) - pd.Timestamp.now().normalize()
).dt.days

# Format execution timestamps
for col in ["cutoff_exec", "exec_12h", "exec_24h", "exec_48h"]:
    forecast_summary[col] = forecast_summary[col].apply(_fmt_dt)

# Lookback availability flags
forecast_summary["has_12h"] = forecast_summary["exec_12h"] != "—"
forecast_summary["has_24h"] = forecast_summary["exec_24h"] != "—"
forecast_summary["has_48h"] = forecast_summary["exec_48h"] != "—"

display_cols = [
    "forecast_date", "days_ahead", "n_regions", "hours_covered",
    "cutoff_exec", "exec_48h", "exec_24h", "exec_12h",
    "has_48h", "has_24h", "has_12h",
    "peak_cutoff_mw", "min_cutoff_mw",
]
forecast_summary[display_cols].style.set_caption(
    "Meteologica — Forecast Date Coverage — 7-Day Forward Window"
).format({
    "peak_cutoff_mw": "{:,.0f}",
    "min_cutoff_mw": "{:,.0f}",
}).set_properties(**{"text-align": "center"})

,forecast_date,days_ahead,n_regions,hours_covered,cutoff_exec,exec_48h,exec_24h,exec_12h,has_48h,has_24h,has_12h,peak_cutoff_mw,min_cutoff_mw
0,2026-03-04,0,4,20,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"100,011","13,143"
1,2026-03-05,1,4,24,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"97,681","12,364"
2,2026-03-06,2,4,24,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"93,737","12,155"
3,2026-03-07,3,4,23,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"87,672","11,972"
4,2026-03-08,4,4,23,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"86,308","12,006"
5,2026-03-09,5,4,24,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"92,676","12,296"
6,2026-03-10,6,4,24,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"94,221","12,470"
7,2026-03-11,7,4,24,2026-03-04 08:22,2026-02-27 12:19,2026-03-03 08:20,2026-03-03 20:20,True,True,True,"96,406","12,541"


## 2. Data Validation — Cutoff Vintage Inspection

In [5]:
# Actual cutoff execution timestamps per forecast_date
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-03-04,2026-03-04 08:22:13,08:22
1,2026-03-05,2026-03-04 08:22:13,08:22
2,2026-03-06,2026-03-04 08:22:13,08:22
3,2026-03-07,2026-03-04 08:22:13,08:22
4,2026-03-08,2026-03-04 08:22:13,08:22
5,2026-03-09,2026-03-04 08:22:13,08:22
6,2026-03-10,2026-03-04 08:22:13,08:22
7,2026-03-11,2026-03-04 08:22:13,08:22


In [6]:
# Bar chart of cutoff hour-of-day
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Meteologica — Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [7]:
# Lookback coverage: which dates have 12h/24h/48h data
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("load_mw_12h", lambda x: x.notna().any()),
        has_24h=("load_mw_24h", lambda x: x.notna().any()),
        has_48h=("load_mw_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-03-04,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
1,2026-03-05,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
2,2026-03-06,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
3,2026-03-07,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
4,2026-03-08,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
5,2026-03-09,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
6,2026-03-10,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37
7,2026-03-11,True,True,True,2026-03-03 20:20:16,2026-03-03 08:20:56,2026-02-27 12:19:37


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [8]:
# For each region, overlay 48h -> 24h -> 12h -> cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].copy()

regions = sorted(df_latest["region"].unique())
fig = make_subplots(
    rows=len(regions), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r} — {latest_date}" for r in regions],
    vertical_spacing=0.06,
)

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

for i, region in enumerate(regions, 1):
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    for label, col, width in [
        ("48h", "load_mw_48h", 1),
        ("24h", "load_mw_24h", 1.5),
        ("12h", "load_mw_12h", 2),
        ("Cutoff", "cutoff_load_mw", 3),
    ]:
        fig.add_trace(
            go.Scatter(
                x=rdf["hour_ending"],
                y=rdf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=300 * len(regions),
    template="plotly_dark",
    title_text=f"Meteologica — Forecast Evolution — Latest Date ({latest_date})",
)
fig.update_yaxes(title_text="Load (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(regions), col=1)
fig.show()

In [9]:
# Same view for all dates — one chart per region, lines per forecast_date
for region in regions:
    rdf = df[df["region"] == region].copy()
    dates = sorted(rdf["forecast_date"].unique())

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=[f"{region} — Cutoff vs 24h Lookback"],
    )

    for j, d in enumerate(dates):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")
        opacity = 0.3 if d != latest_date else 1.0
        # Cutoff
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["cutoff_load_mw"],
            name=f"{d} cutoff",
            line=dict(width=2 if d == latest_date else 1),
            opacity=opacity,
            showlegend=True,
        ))
        # 24h lookback (dashed)
        fig.add_trace(go.Scatter(
            x=ddf["hour_ending"], y=ddf["load_mw_24h"],
            name=f"{d} 24h",
            line=dict(dash="dash", width=1),
            opacity=opacity * 0.6,
            showlegend=False,
        ))

    fig.update_layout(
        height=500, template="plotly_dark",
        title=f"Meteologica {region} — Cutoff (solid) vs 24h Prior (dashed)",
        xaxis_title="Hour Ending", yaxis_title="Load (MW)",
    )
    fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [10]:
# Grouped bar charts: delta_12h, delta_24h, delta_48h by hour_ending for latest date
for region in regions:
    rdf = df_latest[df_latest["region"] == region].sort_values("hour_ending")

    fig = go.Figure()
    for col, label, color in [
        ("delta_48h", "48h\u2192Cutoff", "#636EFA"),
        ("delta_24h", "24h\u2192Cutoff", "#EF553B"),
        ("delta_12h", "12h\u2192Cutoff", "#00CC96"),
    ]:
        fig.add_trace(go.Bar(
            x=rdf["hour_ending"], y=rdf[col],
            name=label, marker_color=color,
        ))

    fig.update_layout(
        barmode="group",
        title=f"Meteologica {region} — Forecast Revision Deltas ({latest_date})",
        xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
        template="plotly_dark", height=400,
    )
    fig.add_hline(y=0, line_color="white", line_width=0.5)
    fig.show()

In [11]:
# Heatmap: 24h delta across all dates x hours, one per region
for region in regions:
    rdf = df[df["region"] == region].copy()
    pivot_24h = rdf.pivot_table(
        index="forecast_date", columns="hour_ending", values="delta_24h",
    )
    if pivot_24h.empty:
        continue

    fig = px.imshow(
        pivot_24h.values,
        x=[f"HE{int(c)}" for c in pivot_24h.columns],
        y=[str(d) for d in pivot_24h.index],
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"Meteologica {region} — 24h Forecast Revision (MW) by Date \u00d7 Hour",
        labels={"color": "Delta MW"},
        template="plotly_dark",
    )
    fig.update_layout(height=500)
    fig.show()

# One subplot row per date, one chart per region
dates = sorted(df["forecast_date"].unique())

for region in regions:
    rdf = df[df["region"] == region].copy()

    fig = make_subplots(
        rows=len(dates), cols=1,
        shared_xaxes=True,
        subplot_titles=[f"{region} — {d}" for d in dates],
        vertical_spacing=0.04,
    )

    for i, d in enumerate(dates, 1):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")

        for label, col, exec_col, width in [
            ("48h", "load_mw_48h", "exec_dt_48h", 1),
            ("24h", "load_mw_24h", "exec_dt_24h", 1.5),
            ("12h", "load_mw_12h", "exec_dt_12h", 2),
            ("Cutoff", "cutoff_load_mw", "cutoff_execution_dt", 3),
        ]:
            exec_dts = ddf[exec_col].astype(str).values
            forecast_dates = ddf["forecast_date"].astype(str).values
            if label == "Cutoff":
                deltas = np.zeros(len(ddf))
            else:
                delta_col_name = f"delta_{label}"
                deltas = ddf[delta_col_name].values
            rank_labels = np.array([label] * len(ddf))
            region_labels = np.array([region] * len(ddf))

            customdata = np.column_stack([exec_dts, forecast_dates, deltas, rank_labels, region_labels])

            fig.add_trace(
                go.Scatter(
                    x=ddf["hour_ending"],
                    y=ddf[col],
                    name=label,
                    line=dict(color=colors[label], dash=dashes[label], width=width),
                    showlegend=(i == 1),
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{customdata[3]}</b> — %{customdata[4]}<br>"
                        "Forecast Date: %{customdata[1]}<br>"
                        "HE: %{x}<br>"
                        "Load: %{y:,.0f} MW<br>"
                        "Execution DT: %{customdata[0]}<br>"
                        "Delta to Cutoff: %{customdata[2]} MW"
                        "<extra></extra>"
                    ),
                ),
                row=i, col=1,
            )

    fig.update_layout(
        height=250 * len(dates),
        template="plotly_dark",
        title_text=f"Meteologica {region} — Forecast Evolution by Date",
    )
    fig.update_yaxes(title_text="Load (MW)")
    fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
    fig.show()

In [12]:
# One subplot row per date, one chart per region
dates = sorted(df["forecast_date"].unique())

for region in regions:
    rdf = df[df["region"] == region].copy()

    fig = make_subplots(
        rows=len(dates), cols=1,
        shared_xaxes=True,
        subplot_titles=[f"{region} — {d}" for d in dates],
        vertical_spacing=0.04,
    )

    for i, d in enumerate(dates, 1):
        ddf = rdf[rdf["forecast_date"] == d].sort_values("hour_ending")

        for label, col, width in [
            ("48h", "load_mw_48h", 1),
            ("24h", "load_mw_24h", 1.5),
            ("12h", "load_mw_12h", 2),
            ("Cutoff", "cutoff_load_mw", 3),
        ]:
            fig.add_trace(
                go.Scatter(
                    x=ddf["hour_ending"],
                    y=ddf[col],
                    name=label,
                    line=dict(color=colors[label], dash=dashes[label], width=width),
                    showlegend=(i == 1),
                ),
                row=i, col=1,
            )

    fig.update_layout(
        height=250 * len(dates),
        template="plotly_dark",
        title_text=f"Meteologica {region} — Forecast Evolution by Date",
    )
    fig.update_yaxes(title_text="Load (MW)")
    fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
    fig.show()

## 6. Revision Summary Statistics

In [13]:
# Summary table: mean, median, std of deltas by region and lookback
summary_rows = []
for region in regions:
    rdf = df[df["region"] == region]
    for lookback, col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
        vals = rdf[col].dropna()
        summary_rows.append({
            "Region": region,
            "Lookback": lookback,
            "Mean (MW)": vals.mean(),
            "Median (MW)": vals.median(),
            "Std (MW)": vals.std(),
            "Min (MW)": vals.min(),
            "Max (MW)": vals.max(),
            "N": len(vals),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Region,Lookback,Mean (MW),Median (MW),Std (MW),Min (MW),Max (MW),N
0,MIDATL,12h,35.000000,49.0,353.104502,-772.0,1376.0,186
1,MIDATL,24h,555.107527,397.5,638.942064,-502.0,2628.0,186
2,MIDATL,48h,-1083.198925,-1074.5,766.881377,-2463.0,1384.0,186
3,RTO,12h,-63.607527,-31.0,511.218197,-1261.0,1511.0,186
4,RTO,24h,843.602151,767.0,940.678089,-1271.0,3847.0,186
5,RTO,48h,-1797.989247,-2051.0,1727.191305,-5204.0,2701.0,186
6,SOUTH,12h,-46.145161,-66.0,108.217526,-302.0,229.0,186
7,SOUTH,24h,31.360215,1.0,212.604783,-524.0,794.0,186
8,SOUTH,48h,-362.188172,-367.5,404.036314,-1344.0,624.0,186
9,WEST,12h,-52.467742,-46.0,310.980092,-845.0,828.0,186


In [14]:
# Per-date summary: avg revision direction across all hours
date_summary = (
    df.groupby(["forecast_date", "region"])
    .agg(
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        mean_delta_48h=("delta_48h", "mean"),
        peak_cutoff_mw=("cutoff_load_mw", "max"),
    )
    .reset_index()
)

# Grouped bar: mean 24h delta per date, faceted by region
fig = px.bar(
    date_summary,
    x="forecast_date",
    y="mean_delta_24h",
    color="region",
    barmode="group",
    title="Meteologica — Mean 24h Forecast Revision (MW) by Date & Region",
    labels={"mean_delta_24h": "Avg 24h Delta (MW)", "forecast_date": "Forecast Date"},
    template="plotly_dark",
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.update_layout(height=450)
fig.show()

In [15]:
# Heatmaps for all regions (12h, 24h, 48h)
for lookback, delta_col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    for region in regions:
        rdf = df[df["region"] == region].copy()
        pivot = rdf.pivot_table(
            index="forecast_date", columns="hour_ending", values=delta_col,
        )
        if pivot.empty:
            continue

        fig = px.imshow(
            pivot.values,
            x=[f"HE{int(c)}" for c in pivot.columns],
            y=[str(d) for d in pivot.index],
            color_continuous_scale="RdBu",
            color_continuous_midpoint=0,
            aspect="auto",
            title=f"Meteologica {region} — {lookback} Forecast Revision (MW) by Date \u00d7 Hour",
            labels={"color": "Delta MW"},
            template="plotly_dark",
        )
        fig.update_layout(height=400)
        fig.show()